# 01 — Data Loading & Exploratory Data Analysis

This notebook:
1. Downloads the UCI Credit Card Default dataset
2. Cleans and standardises column names
3. Saves the cleaned dataset to `data/`
4. Performs EDA: class distribution, histograms, correlation heatmap, default rate by delay status, and summary statistics

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_raw_data, clean_dataframe, save_cleaned_data, load_cleaned_data
from src.utils import setup_logging

setup_logging()
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Load and Clean Data

In [ ]:
# Load raw data (downloads from UCI if not cached)
df_raw = load_raw_data()
print(f'Raw shape: {df_raw.shape}')
df_raw.head()

In [ ]:
# Clean column names and fix category codes
df = clean_dataframe(df_raw)
save_cleaned_data(df)
print(f'Cleaned shape: {df.shape}')
df.head()

## 2. Summary Statistics

In [ ]:
df.describe().T

In [ ]:
df.info()

In [ ]:
# Missing values
print('Missing values per column:')
print(df.isnull().sum())

## 3. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
counts = df['default'].value_counts()
axes[0].bar(['No Default (0)', 'Default (1)'], counts.values,
            color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=['No Default', 'Default'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
            startangle=90, explode=(0, 0.05))
axes[1].set_title('Default Rate')

plt.tight_layout()
plt.show()

## 4. Feature Histograms

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
n_cols = 4
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    df[col].hist(bins=30, ax=axes[i], edgecolor='black', alpha=0.7)
    axes[i].set_title(col, fontsize=10)

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(16, 12))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Default Rate by Repayment Delay Status

In [ ]:
pay_cols = [f'pay_{i}' for i in range(1, 7)]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(pay_cols):
    default_rate = df.groupby(col)['default'].mean()
    axes[i].bar(default_rate.index.astype(str), default_rate.values,
                color='#3498db', edgecolor='black', alpha=0.8)
    axes[i].set_title(f'Default Rate by {col}')
    axes[i].set_ylabel('Default Rate')
    axes[i].set_xlabel(col)
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle('Default Rate by Repayment Delay Status', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 7. Key Observations

- The dataset is **imbalanced**: ~22% defaults vs. ~78% non-defaults.
- Bill amounts are **right-skewed** — most customers have moderate bills.
- Repayment delay status (`pay_1` to `pay_6`) is **strongly correlated** with default.
- Customers with delay status ≥ 2 months have **significantly higher** default rates.